In [1]:
%%capture
#!pip install -U lightautoml
!pip install flaml[automl] matplotlib openml

In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [3]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [4]:
from flaml import AutoML

from flaml.automl.model import LGBMEstimator

In [5]:
N_THREADS = 4
N_FOLDS = 8
RANDOM_STATE = 42
TEST_SIZE = 0.2
TIMEOUT = 3600*100

numpy.random.seed(RANDOM_STATE)
#torch.set_num_threads(N_THREADS)

train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv')
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [6]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [7]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [8]:
train = train.drop(columns=['id'])

train_x = train.drop(columns=['efficiency'])
train_y = train['efficiency']

In [9]:
automl = AutoML()
# best: 1000, holdout
settings = {
    "time_budget": 2000,
    "metric": "mae",
    "estimator_list": ["lgbm"],
    "task": "regression",
    "log_file_name": "experiment.log",
    "seed": 41,
    "eval_method": "cv"
}
automl.fit(X_train = train_x, y_train = train_y, **settings)

[flaml.automl.logger: 05-28 08:12:21] {1752} INFO - task = regression
[flaml.automl.logger: 05-28 08:12:21] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 05-28 08:12:21] {1862} INFO - Minimizing error metric: mae
[flaml.automl.logger: 05-28 08:12:21] {1979} INFO - List of ML learners in AutoML Run: ['lgbm']
[flaml.automl.logger: 05-28 08:12:21] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 05-28 08:12:23] {2417} INFO - Estimated sufficient time budget=15676s. Estimated necessary time budget=16s.
[flaml.automl.logger: 05-28 08:12:23] {2466} INFO -  at 1.7s,	estimator lgbm's best error=0.0843,	best estimator lgbm's best error=0.0843
[flaml.automl.logger: 05-28 08:12:23] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 05-28 08:12:23] {2466} INFO -  at 2.0s,	estimator lgbm's best error=0.0621,	best estimator lgbm's best error=0.0621
[flaml.automl.logger: 05-28 08:12:23] {2282} INFO - iteration 2, current learner lgbm
[flaml.automl.l

In [10]:
#print(f'Concordance index (lifelines): {100*(1-numpy.sqrt(mean_squared_error(y_test, model.predict(X_test))))}')

In [11]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = automl.predict(test)
test_predictions

array([0.39937433, 0.54536559, 0.52672431, ..., 0.61291275, 0.43255892,
       0.5530938 ])

In [12]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions,
})
submission

,id,efficiency
0,0,0.399374
1,1,0.545366
2,2,0.526724
3,3,0.472306
4,4,0.468090
...,...,...
11995,11995,0.547927
11996,11996,0.456874
11997,11997,0.612913
11998,11998,0.432559


In [13]:
submission.to_csv('submission.csv', index = False)